# 01 - Event Study And Payoff Calibration

This notebook reconstructs the PDUFA/FDA event-study logic from the biotech catalyst sleeve. The goal is to estimate approval and CRL payoff templates from abnormal returns, not to claim the sample is large enough for a finished forecasting model.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

from biotech_catalyst.event_payoffs import payoff_distribution, summarise_event_payoffs


In [ ]:
events = pd.read_csv(ROOT / "data" / "pdufa_2025_events.csv")
returns = pd.read_csv(ROOT / "data" / "biotech_event_returns.csv")
price_windows = pd.read_csv(ROOT / "data" / "biotech_price_windows.csv")

display(events.head())
display(returns[["ticker", "decision_date", "outcome", "abnormal_return_5d"]].head())
print("events:", len(events))
print("price-window rows:", len(price_windows))


In [ ]:
summary = summarise_event_payoffs(returns)
summary_df = pd.DataFrame([summary]).T.rename(columns={0: "value"})
display(summary_df.style.format("{:.3f}"))


In [ ]:
payoffs = payoff_distribution(returns)

fig, ax = plt.subplots(figsize=(8, 4))
payoffs.boxplot(column="abnormal_return", by="outcome", ax=ax)
ax.set_title("Approval vs CRL abnormal returns")
ax.set_ylabel("5-day abnormal return")
fig.suptitle("")
plt.show()


In [ ]:
approval_mean = summary["approval_mean"]
crl_mean = summary["crl_mean"]
approval_median = summary["approval_median"]
crl_median = summary["crl_median"]

breakeven_table = pd.DataFrame(
    {
        "template": ["mean", "median"],
        "approval_return": [approval_mean, approval_median],
        "crl_return": [crl_mean, crl_median],
        "breakeven_probability": [
            summary["breakeven_probability_mean"],
            summary["breakeven_probability_median"],
        ],
    }
)

display(breakeven_table.style.format({
    "approval_return": "{:.1%}",
    "crl_return": "{:.1%}",
    "breakeven_probability": "{:.1%}",
}))


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(breakeven_table["template"], breakeven_table["breakeven_probability"])
ax.axhline(summary["approval_rate"], linestyle="--", color="black", label="observed approval rate")
ax.set_ylim(0, 1)
ax.set_ylabel("Probability")
ax.set_title("Break-even approval probability vs observed approval rate")
ax.legend()
plt.show()
